# Build-up vs press — how each team builds, who carries it, how the opponent presses
Barca vs Atletico — first half. **Self-contained: reads only `player_frame_table_named.parquet` and `ball_frame_table_named.parquet`.**

Part 0 recomputes everything the analysis needs from those tables (pressure signal with the v30 thresholds, stoppages, build-up episodes, pressure inside each episode). Nothing is loaded from earlier notebooks.

**Unit of analysis = build-up episode**: one possession of team A inside its own build-up zone (own goal line -> 70% of the pitch). It ends when A escapes past the line, loses the ball, or play stops.

| Part | Question |
|---|---|
| A | **Who** carries the build-up? touches, passes made/received, progression, pressure faced, pass network |
| B | **How** does each team build? start type (goalkeeper or open play), pass mix, lanes, where it escapes and where it loses the ball |
| C | **How does the opponent press it?** where (band x lane), on whom, what triggers it, who presses, man-oriented or zonal, block shape |
| D | Side-by-side summary per team |

**Conventions.** All build-up coordinates are in the *building team's own attack frame*: x = metres from its own goal (it attacks towards larger x), y is flipped for the team that defends x = 105 so left/right stay relative to its own direction of attack. Lanes are thirds of the pitch width: `low-y`, `centre`, `high-y` (which of low-y / high-y is the team's left depends on the camera/pitch orientation in your homography).

# Part 0 — Recompute pressure signal and build-up episodes from the tables

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path

FPS = 25
PITCH_LENGTH = 105
PITCH_WIDTH = 68

CACHE = Path(r"C:\Users\user\Desktop\Football_cv_project\football-pipeline (1)"
             r"\project\barca_atletico_first_half\new_cache")
PLAYER_PATH = CACHE / "player_frame_table_named.parquet"
BALL_PATH = CACHE / "ball_frame_table_named.parquet"

player_frame_table = pd.read_parquet(PLAYER_PATH)
ball_frame_table = pd.read_parquet(BALL_PATH)

# --- adapt to the *_named table structure -------------------------------------
# 1) nullable Int64 -> float64, so <NA> behaves as NaN in comparisons / np.where / groupby
player_frame_table["team"] = player_frame_table["team"].astype("float64")
for c in ["carrier_track_id", "carrier_team", "possession_team"]:
    ball_frame_table[c] = ball_frame_table[c].astype("float64")

# 2) the player table now ALSO carries carrier_track_id / possession_team. The ball table stays the
#    single source of truth (as in v30), so drop the player-side copies -> no _x/_y merge collisions.
player_frame_table = player_frame_table.drop(
    columns=[c for c in ["carrier_track_id", "possession_team"] if c in player_frame_table.columns]
)

# 3) team ids are NOT hard-coded any more. In this cache BAR is team 1 (v30 assumed Barca = team 0),
#    so which team defends x=0 is inferred from the table's own dist_from_own_goal column.
_chk = player_frame_table[
    (player_frame_table["role"] == "player")
    & player_frame_table[["pitch_x", "dist_from_own_goal", "team"]].notna().all(axis=1)
]
_own_goal_at_zero = {}
for _t, _g in _chk.groupby("team"):
    _d_zero = (_g["pitch_x"] - _g["dist_from_own_goal"]).abs().median()
    _d_far = (_g["pitch_x"] + _g["dist_from_own_goal"] - PITCH_LENGTH).abs().median()
    _own_goal_at_zero[int(_t)] = bool(_d_zero < _d_far)
_zero_teams = [t for t, v in _own_goal_at_zero.items() if v]
assert len(_zero_teams) == 1, f"expected exactly one team defending x=0, got {_own_goal_at_zero}"
ZERO_TEAM = _zero_teams[0]   # team whose own goal is at x = 0; the other team defends x = PITCH_LENGTH

def own_goal_distance(team, x):
    # metres from the pressing/defending team's own goal line (v30: np.where(team == 0, x, L - x))
    return np.where(np.asarray(team) == ZERO_TEAM, np.asarray(x, dtype=float), PITCH_LENGTH - np.asarray(x, dtype=float))

TEAM_NAMES = (
    player_frame_table.dropna(subset=["team", "team_name"])
    .groupby("team")["team_name"].agg(lambda s: s.mode().iloc[0])
    .rename(index=int).to_dict()
)

# --- sanity checks -------------------------------------------------------------
print("team ids -> names:", TEAM_NAMES, "| team defending x=0:", ZERO_TEAM, f"({TEAM_NAMES[ZERO_TEAM]})")
print("attack_dir by team:", player_frame_table.dropna(subset=["team", "attack_dir"]).groupby("team")["attack_dir"].unique().to_dict())
print("roles:", player_frame_table["role"].value_counts().to_dict())
print("duplicate (frame_idx, track_id) rows:", int(player_frame_table.duplicated(["frame_idx", "track_id"]).sum()))
print(f"player rows with valid pitch coords: {player_frame_table['pitch_x'].notna().mean():.1%} | "
      f"ball frames with carrier: {ball_frame_table['carrier_track_id'].notna().mean():.1%}")
print("frames:", ball_frame_table["frame_idx"].nunique(), "| player frames:", player_frame_table["frame_idx"].nunique())


## 1. Team defensive baseline
Average depth of each team's back line (last-3 defenders), computed once up front, restricted to frames where that team is out of possession (their actual defending shape, not inflated by attacking-phase positioning). Used in the proximity step to tell "stepped out to press" apart from "block was already sitting there".

In [ ]:
LINE_HEIGHT_DEFENDERS_N = 3

_poss_for_baseline = ball_frame_table[["frame_idx", "possession_team"]]

_outfield_baseline = player_frame_table[player_frame_table["role"] == "player"].dropna(subset=["pitch_x", "pitch_y"]).copy()
_outfield_baseline = _outfield_baseline.merge(_poss_for_baseline, on="frame_idx", how="left")
_outfield_baseline = _outfield_baseline[
    _outfield_baseline["possession_team"].notna() &
    (_outfield_baseline["team"] != _outfield_baseline["possession_team"])
]  # keep only frames where this team is out of possession -- their real defensive shape

_outfield_baseline["dist_from_own_goal"] = own_goal_distance(_outfield_baseline["team"], _outfield_baseline["pitch_x"])
_outfield_baseline["dist_from_own_goal"] = _outfield_baseline["dist_from_own_goal"].clip(0, PITCH_LENGTH)

team_defensive_line_avg = (
    _outfield_baseline.sort_values("dist_from_own_goal")
    .groupby(["frame_idx", "team"])
    .head(LINE_HEIGHT_DEFENDERS_N)
    .groupby("team")["dist_from_own_goal"]
    .mean()
    .to_dict()
)
print("team average defensive line depth, out-of-possession only (last-3 defenders, m from own goal):", team_defensive_line_avg)

## 2. Proximity pressure — distance from each off-ball defender to the ball carrier
A defender only counts as "pressuring" if they're close AND (a) they're closing the distance fast (actively running at the carrier, >=2.5 m/s), or (b) they're within touch-tight range (<1.5m) regardless of speed, or (c) they're clearly ahead of their team's own out-of-possession baseline line (stepped out of the block). This is meant to separate real pressing from a deep, compact block that's simply already sitting near the ball because the attacker walked into it. Thresholds tightened from the first pass -- 1.0 m/s and 2.0m were too permissive and let most of a passive block still count as "pressuring".

In [ ]:
PRESSURE_DIST_THRESH_M = 5.0
CLOSING_WINDOW_FRAMES = 12       # ~0.5s @ 25fps, window used to measure closing speed
CLOSING_SPEED_THRESH_MPS = 2.5   # must be closing at least this fast to count as an active press from open play (walking pace was too permissive)
TIGHT_MARK_DIST_M = 1.5          # inside this range, counts as pressure even without active closing (touch-tight)
STEP_OUT_MARGIN_M = 4.0          # must be at least this far ahead of the team's own average line depth

carrier_pos = ball_frame_table[
    ["frame_idx", "carrier_track_id", "carrier_team", "possession_team", "ball_pitch_x", "ball_pitch_y"]
]

pft = player_frame_table.merge(carrier_pos, on="frame_idx", how="left")

off_ball = pft[
    (pft["role"] == "player") &
    pft["carrier_track_id"].notna() &
    (pft["track_id"] != pft["carrier_track_id"]) &
    (pft["team"] != pft["carrier_team"])
].copy()

off_ball["dist_to_carrier"] = (
    (off_ball["pitch_x"] - off_ball["ball_pitch_x"]) ** 2 +
    (off_ball["pitch_y"] - off_ball["ball_pitch_y"]) ** 2
) ** 0.5

off_ball_pressure = off_ball.dropna(subset=["dist_to_carrier"]).copy()
off_ball_pressure = off_ball_pressure.sort_values(["track_id", "frame_idx"]).reset_index(drop=True)

# closing speed: how fast dist_to_carrier has been shrinking over the last CLOSING_WINDOW_FRAMES for this player
off_ball_pressure["dist_to_carrier_prev"] = off_ball_pressure.groupby("track_id")["dist_to_carrier"].shift(CLOSING_WINDOW_FRAMES)
off_ball_pressure["frame_gap"] = off_ball_pressure["frame_idx"] - off_ball_pressure.groupby("track_id")["frame_idx"].shift(CLOSING_WINDOW_FRAMES)
off_ball_pressure["closing_speed_mps"] = np.where(
    (off_ball_pressure["frame_gap"] > 0) & (off_ball_pressure["frame_gap"] <= CLOSING_WINDOW_FRAMES * 2),
    (off_ball_pressure["dist_to_carrier_prev"] - off_ball_pressure["dist_to_carrier"]) / (off_ball_pressure["frame_gap"] / FPS),
    np.nan
)

# stepped ahead of the team's own habitual defensive line -> not just standing in the block
off_ball_pressure["dist_from_own_goal"] = own_goal_distance(off_ball_pressure["team"], off_ball_pressure["pitch_x"])
off_ball_pressure["dist_from_own_goal"] = off_ball_pressure["dist_from_own_goal"].clip(0, PITCH_LENGTH)
off_ball_pressure["team_line_avg"] = off_ball_pressure["team"].map(team_defensive_line_avg)
off_ball_pressure["stepped_out_of_line"] = (
    off_ball_pressure["dist_from_own_goal"] >= off_ball_pressure["team_line_avg"] + STEP_OUT_MARGIN_M
)

off_ball_pressure["is_pressuring"] = (
    (off_ball_pressure["dist_to_carrier"] < PRESSURE_DIST_THRESH_M) &
    (
        (off_ball_pressure["dist_to_carrier"] < TIGHT_MARK_DIST_M) |
        (off_ball_pressure["closing_speed_mps"].fillna(0) >= CLOSING_SPEED_THRESH_MPS) |
        off_ball_pressure["stepped_out_of_line"]
    )
)

team_pressure_per_frame = (
    off_ball_pressure.groupby(["frame_idx", "team", "carrier_team", "possession_team"], as_index=False)
    .agg(
        n_pressing=("is_pressuring", "sum"),
        n_defenders_on_pitch=("track_id", "nunique"),
        min_dist_to_carrier=("dist_to_carrier", "min"),
    )
)
team_pressure_per_frame["pressing_intensity"] = (
    team_pressure_per_frame["n_pressing"] / team_pressure_per_frame["n_defenders_on_pitch"]
)

## 3. Passing-lane pressure — how covered are the carrier's passing options

In [ ]:
MIN_LANE_LENGTH_M = 3.0
TIGHT_LANE_THRESH_M = 1.5
CONTESTED_LANE_THRESH_M = 3.0

carrier_frames = pft[(pft["role"] == "player") & pft["carrier_track_id"].notna()].copy()

teammates = carrier_frames[
    (carrier_frames["team"] == carrier_frames["carrier_team"]) &
    (carrier_frames["track_id"] != carrier_frames["carrier_track_id"])
]
opponents = carrier_frames[carrier_frames["team"] != carrier_frames["carrier_team"]]

carrier_player_pos = (
    player_frame_table[player_frame_table["is_carrier"] == True]
    [["frame_idx", "track_id", "pitch_x", "pitch_y"]]
    .rename(columns={"track_id": "carrier_track_id", "pitch_x": "carrier_x", "pitch_y": "carrier_y"})
    .drop_duplicates(subset="frame_idx", keep="first")
)

teammate_pos = teammates[["frame_idx", "track_id", "pitch_x", "pitch_y"]].rename(
    columns={"track_id": "teammate_id", "pitch_x": "tm_x", "pitch_y": "tm_y"}
)
opponent_pos = opponents[["frame_idx", "track_id", "pitch_x", "pitch_y"]].rename(
    columns={"track_id": "opponent_id", "pitch_x": "opp_x", "pitch_y": "opp_y"}
)

lanes = teammate_pos.merge(opponent_pos, on="frame_idx", how="inner")
lanes = lanes.merge(carrier_player_pos, on="frame_idx", how="inner")
lanes = lanes.dropna(subset=["carrier_x", "carrier_y"])

Ax, Ay = lanes["carrier_x"].values, lanes["carrier_y"].values
Bx, By = lanes["tm_x"].values, lanes["tm_y"].values
Px, Py = lanes["opp_x"].values, lanes["opp_y"].values

ABx, ABy = Bx - Ax, By - Ay
APx, APy = Px - Ax, Py - Ay

seg_len_sq = ABx ** 2 + ABy ** 2
seg_len_sq = np.where(seg_len_sq == 0, 1e-9, seg_len_sq)

t = (APx * ABx + APy * ABy) / seg_len_sq
t_clipped = np.clip(t, 0.0, 1.0)

closest_x = Ax + t_clipped * ABx
closest_y = Ay + t_clipped * ABy
perp_dist = np.sqrt((Px - closest_x) ** 2 + (Py - closest_y) ** 2)

lanes["t"] = t
lanes["lane_perp_dist"] = perp_dist
lanes["lane_length_m"] = np.sqrt(seg_len_sq)

valid_lanes = lanes[
    (lanes["lane_length_m"] >= MIN_LANE_LENGTH_M) &
    (lanes["t"] >= 0.15) &
    (lanes["t"] <= 0.85)
].copy()

valid_lanes["lane_tight_block"] = valid_lanes["lane_perp_dist"] < TIGHT_LANE_THRESH_M
valid_lanes["lane_contested"] = valid_lanes["lane_perp_dist"] < CONTESTED_LANE_THRESH_M

lane_status = (
    valid_lanes.groupby(["frame_idx", "teammate_id"], as_index=False)
    .agg(lane_tight_block=("lane_tight_block", "any"), lane_contested=("lane_contested", "any"))
)

frame_lane_summary = (
    lane_status.groupby("frame_idx", as_index=False)
    .agg(
        n_lanes_total=("teammate_id", "nunique"),
        n_lanes_tight_blocked=("lane_tight_block", "sum"),
        n_lanes_contested=("lane_contested", "sum"),
    )
)
frame_lane_summary["n_lanes_open"] = frame_lane_summary["n_lanes_total"] - frame_lane_summary["n_lanes_contested"]
frame_lane_summary["frac_tight_blocked"] = frame_lane_summary["n_lanes_tight_blocked"] / frame_lane_summary["n_lanes_total"]
frame_lane_summary["frac_contested"] = frame_lane_summary["n_lanes_contested"] / frame_lane_summary["n_lanes_total"]

frame_lane_summary = frame_lane_summary.merge(
    carrier_pos[["frame_idx", "carrier_team", "possession_team"]], on="frame_idx", how="left"
)


## 4. Unified team pressure signal + events
Combines proximity pressure OR lane-blocking pressure, then excludes own-penalty-box frames (16.5m from own goal) since compact box defending is a different phenomenon from active pressing. This corrected `under_team_pressure` definition is used everywhere downstream (events, zone breakdown, time-to-press, pass resistance, weighted intensity).
Note: `n_pressing`/`pressing_intensity` now come from the closing-speed + step-out-of-line gated definition in section 1, not raw distance -- so a passive deep block no longer inflates this on its own. The lane-blocking signal (section 2) is left distance-based for now, since being within `TIGHT_LANE_THRESH_M` of an actual pass lane is a real obstruction regardless of how the defender got there.

In [ ]:
N_PRESSING_THRESH = 2
FRAC_TIGHT_BLOCKED_THRESH = 0.375
MIN_EVENT_DURATION_SEC = 0.5
GAP_TOLERANCE_FRAMES = 5
BOX_DEPTH_M = 16.5

team_pressure_full = team_pressure_per_frame.merge(
    frame_lane_summary[["frame_idx", "n_lanes_total", "n_lanes_tight_blocked",
                         "n_lanes_contested", "frac_tight_blocked", "frac_contested"]],
    on="frame_idx",
    how="left"
)

ball_pos_by_frame = ball_frame_table[["frame_idx", "ball_pitch_x", "ball_pitch_y"]].rename(
    columns={"ball_pitch_x": "press_x", "ball_pitch_y": "press_y"}
)
team_pressure_full = team_pressure_full.merge(ball_pos_by_frame, on="frame_idx", how="left")

tpf = team_pressure_full.copy()
tpf["frac_tight_blocked"] = tpf["frac_tight_blocked"].fillna(0)

tpf["in_own_box"] = own_goal_distance(tpf["team"], tpf["press_x"]) <= BOX_DEPTH_M

tpf["under_team_pressure"] = (
    ((tpf["n_pressing"] >= N_PRESSING_THRESH) | (tpf["frac_tight_blocked"] >= FRAC_TIGHT_BLOCKED_THRESH)) &
    (~tpf["in_own_box"])
)

# carry the corrected flag back onto team_pressure_full so every downstream section uses the same definition
team_pressure_full["under_team_pressure"] = tpf["under_team_pressure"]

tpf = tpf.sort_values(["team", "frame_idx"]).reset_index(drop=True)

events_list = []
for team_id, grp in tpf.groupby("team"):
    grp = grp.sort_values("frame_idx").reset_index(drop=True)

    is_press = grp["under_team_pressure"].values.copy()
    frames = grp["frame_idx"].values
    i, n = 0, len(is_press)
    while i < n:
        if not is_press[i]:
            j = i
            while j < n and not is_press[j]:
                j += 1
            gap_frame_span = frames[j - 1] - frames[i] + 1 if j > i else 0
            if i > 0 and j < n and gap_frame_span <= GAP_TOLERANCE_FRAMES:
                is_press[i:j] = True
            i = j
        else:
            i += 1
    grp["under_team_pressure_bridged"] = is_press

    grp["state_change"] = (grp["under_team_pressure_bridged"] != grp["under_team_pressure_bridged"].shift()).cumsum()
    runs = (
        grp.groupby("state_change")
        .agg(
            start_frame=("frame_idx", "first"),
            end_frame=("frame_idx", "last"),
            n_frames=("frame_idx", "count"),
            under_pressure=("under_team_pressure_bridged", "first"),
            possession_team=("possession_team", "first"),
            mean_n_pressing=("n_pressing", "mean"),
            mean_frac_tight_blocked=("frac_tight_blocked", "mean"),
        )
        .reset_index(drop=True)
    )
    runs["team"] = team_id
    events_list.append(runs)

team_pressure_events = pd.concat(events_list, ignore_index=True)
team_pressure_events = team_pressure_events[team_pressure_events["under_pressure"]].copy()
team_pressure_events["duration_sec"] = team_pressure_events["n_frames"] / FPS
team_pressure_events = team_pressure_events[
    team_pressure_events["duration_sec"] >= MIN_EVENT_DURATION_SEC
].reset_index(drop=True)

print("total team pressure events:", len(team_pressure_events))
print(team_pressure_events.groupby("team")["duration_sec"].describe())


### Pressure signal ready
`team_pressure_full` (per frame, per defending team) and `team_pressure_events` (v30 events) are now computed from the tables above. Team ids are cast to int for the joins below.

In [ ]:
team_pressure_events["team"] = team_pressure_events["team"].astype(int)
team_pressure_full["team"] = team_pressure_full["team"].astype(int)
print(f"{len(team_pressure_events)} pressure events | {len(team_pressure_full)} per-frame pressure rows")

## Stoppages (v30, unchanged)

In [ ]:
STOPPAGE_GAP_SEC = 10.0
STOPPAGE_GAP_FRAMES = int(STOPPAGE_GAP_SEC * FPS)

carrier_missing = ball_frame_table[["frame_idx", "carrier_track_id"]].sort_values("frame_idx").reset_index(drop=True)
carrier_missing["is_missing"] = carrier_missing["carrier_track_id"].isna()
carrier_missing["gap_id"] = (carrier_missing["is_missing"] != carrier_missing["is_missing"].shift()).cumsum()

gap_runs = (
    carrier_missing[carrier_missing["is_missing"]]
    .groupby("gap_id")
    .agg(start_frame=("frame_idx", "first"), end_frame=("frame_idx", "last"), n_frames=("frame_idx", "count"))
    .reset_index(drop=True)
)
gap_runs["duration_sec"] = gap_runs["n_frames"] / FPS

stoppages = gap_runs[gap_runs["duration_sec"] >= STOPPAGE_GAP_SEC].reset_index(drop=True)

# vectorized frame -> is_stoppage lookup
_stoppage_starts = stoppages["start_frame"].values
_stoppage_ends = stoppages["end_frame"].values

all_frames = ball_frame_table[["frame_idx"]].drop_duplicates().sort_values("frame_idx").reset_index(drop=True)
_frame_vals = all_frames["frame_idx"].values
_idx = np.searchsorted(_stoppage_starts, _frame_vals, side="right") - 1
_valid = _idx >= 0
_in_range = np.zeros(len(_frame_vals), dtype=bool)
_idx_valid = _idx[_valid]
_in_range[_valid] = _frame_vals[_valid] <= _stoppage_ends[_idx_valid]
all_frames["is_stoppage"] = _in_range

stoppage_frame_lookup = all_frames.set_index("frame_idx")["is_stoppage"]

print(f"{len(stoppages)} stoppages >= {STOPPAGE_GAP_SEC}s flagged, covering {stoppages['n_frames'].sum()} frames "
      f"({stoppages['duration_sec'].sum():.1f}s total dead time)")

## Episode config + per-frame state
`possession_team` is forward-filled over no-carrier gaps (same keep-last rule as v30 section 12), so a ball in flight stays with the passer until someone else actually gets it.

In [ ]:
ZONE_FRAC = 0.70
ZONE_M = ZONE_FRAC * PITCH_LENGTH          # 73.5 m from the build-up team's own goal
HOLD_SEC = 0.5                             # escape / loss must persist this long
HOLD_FRAMES = int(HOLD_SEC * FPS)
MIN_EPISODE_SEC = 1.0
REENTRY_MARGIN_M = 3.0                     # after `escaped`, ball must return to <= ZONE_M - this before a new episode can start
BALL_FFILL_FRAMES = FPS                    # bridge <= 1 s of missing ball position

ball = (
    ball_frame_table[["frame_idx", "possession_team", "ball_pitch_x"]]
    .sort_values("frame_idx").reset_index(drop=True)
    .merge(stoppage_frame_lookup.rename("is_stoppage"), left_on="frame_idx", right_index=True, how="left")
)
ball["is_stoppage"] = ball["is_stoppage"].fillna(False).astype(bool)
ball["poss"] = ball["possession_team"].ffill()
ball["ball_x"] = ball["ball_pitch_x"].ffill(limit=BALL_FFILL_FRAMES)

print(f"zone line: {ZONE_M:.1f} m from each team's own goal")
print("non-contiguous frame steps:", int((ball["frame_idx"].diff().dropna() != 1).sum()))
print(f"ball position known: {ball['ball_x'].notna().mean():.1%} | possession known: {ball['poss'].notna().mean():.1%}")

## Detect episodes

In [ ]:
def detect_episodes(team):
    opp = [t for t in TEAM_NAMES if t != team][0]
    frames = ball["frame_idx"].values
    poss = ball["poss"].values
    stop = ball["is_stoppage"].values
    dist = own_goal_distance(np.full(len(ball), team), ball["ball_x"].values)

    out, in_ep, need_reentry = [], False, False
    s = esc_run = loss_run = esc_start = loss_start = 0

    def close(e, outcome):
        out.append(dict(
            team=team, defending_team=opp,
            start_frame=int(frames[s]), end_frame=int(frames[e]),
            n_frames=int(frames[e] - frames[s] + 1), outcome=outcome,
            start_dist_m=float(dist[s]), max_dist_m=float(np.nanmax(dist[s:e + 1])),
        ))

    for i in range(len(ball)):
        if stop[i]:
            if in_ep:
                close(i - 1, "stopped"); in_ep = False
            need_reentry = False
            continue
        a_has, o_has = poss[i] == team, poss[i] == opp
        if need_reentry and dist[i] <= ZONE_M - REENTRY_MARGIN_M:
            need_reentry = False
        if not in_ep:
            if a_has and dist[i] <= ZONE_M and not need_reentry:
                in_ep, s, esc_run, loss_run = True, i, 0, 0
            continue
        beyond = a_has and dist[i] > ZONE_M
        esc_run = esc_run + 1 if beyond else 0
        loss_run = loss_run + 1 if o_has else 0
        if esc_run == 1: esc_start = i
        if loss_run == 1: loss_start = i
        if esc_run >= HOLD_FRAMES:
            close(esc_start - 1, "escaped"); in_ep = False; need_reentry = True
        elif loss_run >= HOLD_FRAMES:
            close(loss_start - 1, "lost"); in_ep = False
    if in_ep:
        close(len(ball) - 1, "data_end")
    return out

episodes = pd.DataFrame([e for t in TEAM_NAMES for e in detect_episodes(t)])
n_raw = len(episodes)
episodes["duration_sec"] = episodes["n_frames"] / FPS
episodes = episodes[episodes["duration_sec"] >= MIN_EPISODE_SEC].sort_values("start_frame").reset_index(drop=True)
episodes.insert(0, "episode_id", episodes.index)
episodes["team_name"] = episodes["team"].map(TEAM_NAMES)
episodes["defending_team_name"] = episodes["defending_team"].map(TEAM_NAMES)
_mmss = lambda sec: f"{int(sec // 60):02d}:{int(sec % 60):02d}"
episodes["start_mmss"] = (episodes["start_frame"] / FPS).map(_mmss)
episodes["end_mmss"] = ((episodes["end_frame"] + 1) / FPS).map(_mmss)

print(f"episodes starting within {REENTRY_MARGIN_M:.0f} m of the line: {int((episodes['start_dist_m'] > ZONE_M - REENTRY_MARGIN_M).sum())}")
print(f"{n_raw} raw episodes -> {len(episodes)} after dropping < {MIN_EPISODE_SEC}s")
display(episodes.groupby("team_name").agg(
    n_episodes=("episode_id", "count"),
    mean_sec=("duration_sec", "mean"),
    median_sec=("duration_sec", "median"),
    max_sec=("duration_sec", "max"),
).round(2))
display(episodes.groupby(["team_name", "outcome"]).size().unstack(fill_value=0))

## Pressure inside each episode
- `pressed` / `n_press_events` / `pressed_sec` / `time_to_first_press_sec`: overlap with the defending team's v30 pressure events.
- `pressed_active`: >= 0.5 s of proximity-driven frames (`n_pressing >= 2`), i.e. defenders actually closing down, not just covering lanes.
- `pressed_first_3s` / `pressed_last_2s`: v30 pressure overlapping the first 3 s / last 2 s of the episode. These windows are the same length for every episode, so they don't favour long episodes.
- `raw_frames_*`: per-frame breakdown of *why* a frame counted as pressure (proximity = `n_pressing >= 2`, lane = `frac_tight_blocked >= 0.375`). Frames with no ball carrier have no pressure row and count as not pressed.

In [ ]:
# --- v30 event overlap ---
ev_by_team = {t: g[["start_frame", "end_frame"]].values for t, g in team_pressure_events.groupby("team")}

FIRST_WINDOW_SEC, LAST_WINDOW_SEC = 3.0, 2.0
MIN_ACTIVE_FRAMES = int(0.5 * FPS)     # 0.5 s, same as v30 MIN_EVENT_DURATION_SEC

def _overlap_frames(evs, lo, hi):
    if len(evs) == 0 or hi < lo:
        return 0
    a = np.maximum(evs[:, 0], lo)
    b = np.minimum(evs[:, 1], hi)
    return int(np.clip(b - a + 1, 0, None).sum())

def event_overlap(row):
    evs = ev_by_team.get(row.defending_team, np.empty((0, 2), dtype=int))
    hit = evs[(evs[:, 1] >= row.start_frame) & (evs[:, 0] <= row.end_frame)]
    first_hi = min(row.end_frame, row.start_frame + int(FIRST_WINDOW_SEC * FPS) - 1)
    last_lo = max(row.start_frame, row.end_frame - int(LAST_WINDOW_SEC * FPS) + 1)
    out = dict(
        press_frames_first_3s=_overlap_frames(evs, row.start_frame, first_hi),
        press_frames_last_2s=_overlap_frames(evs, last_lo, row.end_frame),
    )
    if len(hit) == 0:
        return pd.Series(dict(n_press_events=0, pressed_frames=0, first_press_frame=np.nan, **out))
    lo = np.maximum(hit[:, 0], row.start_frame)
    hi = np.minimum(hit[:, 1], row.end_frame)
    return pd.Series(dict(n_press_events=len(hit), pressed_frames=int((hi - lo + 1).sum()), first_press_frame=float(lo.min()), **out))

episodes = episodes.drop(columns=["n_press_events", "pressed_frames", "first_press_frame", "press_frames_first_3s", "press_frames_last_2s"], errors="ignore")
episodes = episodes.join(episodes.apply(event_overlap, axis=1))
_ic = ["n_press_events", "pressed_frames", "press_frames_first_3s", "press_frames_last_2s"]
episodes[_ic] = episodes[_ic].astype(int)
episodes["pressed"] = episodes["n_press_events"] > 0
episodes["pressed_first_3s"] = episodes["press_frames_first_3s"] > 0
episodes["pressed_last_2s"] = episodes["press_frames_last_2s"] > 0
episodes["pressed_sec"] = episodes["pressed_frames"] / FPS
episodes["pct_time_pressed"] = episodes["pressed_frames"] / episodes["n_frames"]
episodes["time_to_first_press_sec"] = (episodes["first_press_frame"] - episodes["start_frame"]) / FPS

# --- per-frame breakdown ---
ep_frames = pd.concat(
    [pd.DataFrame({"episode_id": r.episode_id, "defending_team": r.defending_team,
                   "frame_idx": np.arange(r.start_frame, r.end_frame + 1)}) for r in episodes.itertuples()],
    ignore_index=True,
)
tp = team_pressure_full[["frame_idx", "team", "n_pressing", "frac_tight_blocked", "under_team_pressure"]].copy()
tp["frac_tight_blocked"] = tp["frac_tight_blocked"].fillna(0)
ep_frames = ep_frames.merge(tp, left_on=["frame_idx", "defending_team"], right_on=["frame_idx", "team"], how="left")

under = ep_frames["under_team_pressure"].fillna(False).astype(bool)
prox = ep_frames["n_pressing"].fillna(0) >= N_PRESSING_THRESH
lane = ep_frames["frac_tight_blocked"] >= FRAC_TIGHT_BLOCKED_THRESH
ep_frames["f_under"] = under
ep_frames["f_prox_only"] = under & prox & ~lane
ep_frames["f_lane_only"] = under & ~prox & lane
ep_frames["f_both"] = under & prox & lane

frame_stats = ep_frames.groupby("episode_id").agg(
    raw_frames_under_pressure=("f_under", "sum"),
    raw_frames_prox_only=("f_prox_only", "sum"),
    raw_frames_lane_only=("f_lane_only", "sum"),
    raw_frames_both=("f_both", "sum"),
    peak_n_pressing=("n_pressing", "max"),
    frames_with_carrier=("n_pressing", "count"),
).reset_index()
episodes = episodes.drop(columns=[c for c in frame_stats.columns if c != "episode_id"], errors="ignore").merge(frame_stats, on="episode_id", how="left")

episodes["raw_frames_active"] = episodes["raw_frames_prox_only"] + episodes["raw_frames_both"]
episodes["pressed_active"] = episodes["raw_frames_active"] >= MIN_ACTIVE_FRAMES

print(f"{int(episodes['pressed'].sum())} / {len(episodes)} episodes pressed | {int(episodes['pressed_active'].sum())} actively pressed (proximity-driven)")

## Shared helpers
Identities (the tables are named: `track_id` is a player identity), team-frame coordinates, lane and distance bands, per-frame pressure lookup.

In [ ]:
# ---------- identities ----------
def _mode(s):
    s = s.dropna()
    return s.mode().iloc[0] if len(s) else np.nan

_pi = player_frame_table[player_frame_table["role"] != "referee"]
id_info = _pi.groupby("track_id").agg(name=("name", _mode), number=("number", _mode), role=("role", _mode)).reset_index()

def _label(r):
    num = "" if pd.isna(r["number"]) else f"#{int(float(r['number']))} "
    nm = "?" if pd.isna(r["name"]) else str(r["name"])
    return f"{num}{nm}"

id_info["label"] = id_info.apply(_label, axis=1)
LABEL = id_info.set_index("track_id")["label"].to_dict()
NUMBER = id_info.set_index("track_id")["number"].to_dict()
IS_GK = id_info.set_index("track_id")["role"].eq("goalkeeper").to_dict()
print("roles in the named table:", id_info["role"].value_counts().to_dict(),
      "| goalkeeper ids:", [LABEL[k] for k, v in IS_GK.items() if v])

opp_of = {t: [u for u in TEAM_NAMES if u != t][0] for t in TEAM_NAMES}
TEAM_COLORS = dict(zip(sorted(TEAM_NAMES), ["#1f6fb2", "#d9442b"]))
BUILD_TEAMS = sorted(TEAM_NAMES)

# ---------- team attack frame ----------
def to_team_frame(team, x, y):
    """x = metres from the team's own goal; y flipped for the team that defends x=PITCH_LENGTH (180-degree rotation)."""
    team = np.asarray(team); x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    flip = team != ZERO_TEAM
    return np.where(flip, PITCH_LENGTH - x, x), np.where(flip, PITCH_WIDTH - y, y)

LANES = ["low-y", "centre", "high-y"]
def lane_of(y):
    y = np.asarray(y, dtype=float)
    out = np.where(y < PITCH_WIDTH / 3, "low-y", np.where(y > 2 * PITCH_WIDTH / 3, "high-y", "centre")).astype(object)
    out[np.isnan(y)] = None
    return out

BAND_EDGES = [-np.inf, 16.5, 35.0, 52.5, np.inf]
BANDS = ["0-16.5 (box)", "16.5-35", "35-52.5", "52.5+"]
def band_of(x):
    return np.asarray(pd.cut(np.asarray(x, dtype=float), BAND_EDGES, labels=BANDS).astype(object))

# ---------- per-frame "defending team is pressing" lookup (v30 under_team_pressure, defending team = key) ----------
N_FRAMES_TOTAL = int(ball_frame_table["frame_idx"].max()) + 2
under_arr = {t: np.zeros(N_FRAMES_TOTAL, dtype=bool) for t in TEAM_NAMES}
_u = team_pressure_full[team_pressure_full["under_team_pressure"].fillna(False).astype(bool)]
for _t, _g in _u.groupby("team"):
    under_arr[int(_t)][_g["frame_idx"].values.astype(int)] = True

MIN_SUSTAINED_PRESSURE_FRAMES = 8   # same as v30 section 10
def sustained(arr, f0, f1, n=MIN_SUSTAINED_PRESSURE_FRAMES):
    seg = arr[max(0, f0):f1 + 1]
    if seg.sum() < n:
        return False
    d = np.diff(np.concatenate(([0], seg.astype(int), [0])))
    st, en = np.where(d == 1)[0], np.where(d == -1)[0]
    return bool(len(st) and (en - st).max() >= n)

# ---------- ball position per frame ----------
ball_xy = (ball_frame_table.sort_values("frame_idx").drop_duplicates("frame_idx")
           .set_index("frame_idx")[["ball_pitch_x", "ball_pitch_y"]].ffill(limit=FPS))

# ---------- pitch drawing ----------
import matplotlib.patches as patches
def draw_pitch(ax, zone=True):
    L, W = PITCH_LENGTH, PITCH_WIDTH
    ax.add_patch(patches.Rectangle((0, 0), L, W, fill=False, lw=1.2, ec="#444"))
    ax.plot([L / 2, L / 2], [0, W], c="#444", lw=1)
    for x0 in (0, L - 16.5):
        ax.add_patch(patches.Rectangle((x0, (W - 40.32) / 2), 16.5, 40.32, fill=False, lw=1, ec="#444"))
    if zone:
        ax.axvline(ZONE_M, ls="--", c="#888", lw=1)
    ax.set_xlim(-2, L + 2); ax.set_ylim(-2, W + 2); ax.set_aspect("equal"); ax.axis("off")

print("helpers ready")

## Build-up events: carrier runs, touches, passes
Same detection as v30 section 10, restricted to **the building team's own episodes**: a carrier run is a stretch of frames with the same `carrier_track_id` (>= 2 frames); consecutive runs form a pass attempt (same team = completed, other team = intercepted), pairs closer than 3 m are treated as a duel, not a pass, and gaps over 2 s are skipped.
- direction: `forward` / `backward` if the pass changes distance-from-own-goal by >= 3 m, else `lateral`
- length: `long` >= 30 m, `progressive` = completed and >= 10 m forward
- `under_pressure` = the opponent's v30 pressure flag holds for >= 8 consecutive frames in the 1 s before release

In [ ]:
MIN_HOLD_FRAMES_PASS = 2
MIN_PASS_DISTANCE_M = 3.0
MAX_RECEPTION_GAP_FRAMES = 2 * FPS
DIRECTION_THRESH_M = 3.0
LONG_PASS_M = 30.0
PROGRESSIVE_M = 10.0
PRESSURE_LOOKBACK_FRAMES = FPS

# ---- carrier runs ----
carrier_seq = (ball_frame_table.dropna(subset=["carrier_track_id", "carrier_team"])
               [["frame_idx", "carrier_track_id", "carrier_team"]].sort_values("frame_idx").reset_index(drop=True))
carrier_seq["carrier_track_id"] = carrier_seq["carrier_track_id"].astype(int)
carrier_seq["carrier_team"] = carrier_seq["carrier_team"].astype(int)
carrier_seq["run_id"] = (carrier_seq["carrier_track_id"] != carrier_seq["carrier_track_id"].shift()).cumsum()
runs = (carrier_seq.groupby("run_id")
        .agg(track_id=("carrier_track_id", "first"), team=("carrier_team", "first"),
             start_frame=("frame_idx", "first"), end_frame=("frame_idx", "last"), n_frames=("frame_idx", "count"))
        .reset_index(drop=True))
runs = runs[runs["n_frames"] >= MIN_HOLD_FRAMES_PASS].reset_index(drop=True)

# ---- player positions at run boundaries (vectorised lookup) ----
pos_lookup = (player_frame_table.dropna(subset=["pitch_x", "pitch_y"])
              .drop_duplicates(["frame_idx", "track_id"]).set_index(["frame_idx", "track_id"])[["pitch_x", "pitch_y"]].sort_index())

def positions_at(frames, tracks):
    q = pd.MultiIndex.from_arrays([np.asarray(frames, dtype=int), np.asarray(tracks, dtype=int)])
    idx = pos_lookup.index.get_indexer(q)
    out = np.full((len(idx), 2), np.nan)
    ok = idx >= 0
    out[ok] = pos_lookup.values[idx[ok]]
    return out

_s = positions_at(runs["start_frame"], runs["track_id"])
_e = positions_at(runs["end_frame"], runs["track_id"])
runs["start_x_att"], runs["start_y_att"] = to_team_frame(runs["team"].values, _s[:, 0], _s[:, 1])
runs["end_x_att"], runs["end_y_att"] = to_team_frame(runs["team"].values, _e[:, 0], _e[:, 1])
runs["_sx"], runs["_sy"], runs["_ex"], runs["_ey"] = _s[:, 0], _s[:, 1], _e[:, 0], _e[:, 1]

# ---- which episode (of the carrier's own team) does a frame belong to ----
_ep_arr = {t: g.sort_values("start_frame")[["episode_id", "start_frame", "end_frame"]].values for t, g in episodes.groupby("team")}
def episode_at(team, frame):
    arr = _ep_arr.get(int(team))
    if arr is None or len(arr) == 0:
        return -1
    i = np.searchsorted(arr[:, 1], frame, side="right") - 1
    return int(arr[i, 0]) if i >= 0 and frame <= arr[i, 2] else -1

runs["episode_id"] = [episode_at(t, f) for t, f in zip(runs["team"], runs["start_frame"])]

# ---- passes from consecutive runs ----
gap = runs["start_frame"].values[1:] - runs["end_frame"].values[:-1]
passes = pd.DataFrame({
    "passer_id": runs["track_id"].values[:-1], "passer_team": runs["team"].values[:-1],
    "receiver_id": runs["track_id"].values[1:], "receiver_team": runs["team"].values[1:],
    "release_frame": runs["end_frame"].values[:-1], "reception_frame": runs["start_frame"].values[1:],
    "px": runs["_ex"].values[:-1], "py": runs["_ey"].values[:-1], "rx": runs["_sx"].values[1:], "ry": runs["_sy"].values[1:],
})
passes = passes[(gap <= MAX_RECEPTION_GAP_FRAMES)].dropna(subset=["px", "py", "rx", "ry"]).reset_index(drop=True)
passes["length_m"] = np.hypot(passes["rx"] - passes["px"], passes["ry"] - passes["py"])
passes["completed"] = passes["passer_team"] == passes["receiver_team"]
passes["receiver_is_gk"] = passes["receiver_id"].map(IS_GK).fillna(False).astype(bool)
passes = passes[passes["length_m"] >= MIN_PASS_DISTANCE_M]
passes = passes[~((~passes["completed"]) & passes["receiver_is_gk"])].reset_index(drop=True)   # keeper save/claim, not a pass

passes["episode_id"] = [episode_at(t, f) for t, f in zip(passes["passer_team"], passes["release_frame"])]
passes = passes[passes["episode_id"] >= 0].reset_index(drop=True)          # build-up passes only

passes["rel_x_att"], passes["rel_y_att"] = to_team_frame(passes["passer_team"].values, passes["px"], passes["py"])
passes["rec_x_att"], passes["rec_y_att"] = to_team_frame(passes["passer_team"].values, passes["rx"], passes["ry"])
passes["progression_m"] = passes["rec_x_att"] - passes["rel_x_att"]
passes["direction"] = np.where(passes["progression_m"] >= DIRECTION_THRESH_M, "forward",
                        np.where(passes["progression_m"] <= -DIRECTION_THRESH_M, "backward", "lateral"))
passes["is_long"] = passes["length_m"] >= LONG_PASS_M
passes["progressive_completed"] = passes["completed"] & (passes["progression_m"] >= PROGRESSIVE_M)
passes["rel_lane"], passes["rec_lane"] = lane_of(passes["rel_y_att"]), lane_of(passes["rec_y_att"])
passes["rel_band"], passes["rec_band"] = band_of(passes["rel_x_att"]), band_of(passes["rec_x_att"])
passes["passer_is_gk"] = passes["passer_id"].map(IS_GK).fillna(False).astype(bool)
passes["passer_label"] = passes["passer_id"].map(LABEL)
passes["receiver_label"] = passes["receiver_id"].map(LABEL)
passes["under_pressure"] = [
    sustained(under_arr[opp_of[int(t)]], int(f) - PRESSURE_LOOKBACK_FRAMES, int(f))
    for t, f in zip(passes["passer_team"], passes["release_frame"])
]

# ---- run-level pressure + carry progression ----
runs_ep = runs[runs["episode_id"] >= 0].copy()
runs_ep["press_frames"] = [int(under_arr[opp_of[int(t)]][s:e + 1].sum()) for t, s, e in zip(runs_ep["team"], runs_ep["start_frame"], runs_ep["end_frame"])]
runs_ep["carry_prog_m"] = np.clip(runs_ep["end_x_att"] - runs_ep["start_x_att"], 0, None).fillna(0)

print(f"{len(runs_ep)} carrier runs and {len(passes)} build-up passes inside {len(episodes)} episodes")
print(passes.groupby("passer_team").agg(attempts=("completed", "size"), completed=("completed", "sum"),
                                         under_pressure=("under_pressure", "mean")).rename(index=TEAM_NAMES).round(2))

# Part A — Who helps the build-up
Per player, inside the team's own build-up episodes:
- `touches` = carrier runs (every reception, regain or restart that turns into a hold), `touch_share` = share of the team's touches
- `passes_made` / `passes_received`, completion, progressive passes (>= 10 m forward), long passes (>= 30 m)
- `carry_prog_m` = metres gained forward while carrying
- `pct_on_ball_pressed` = share of the time on the ball that the opponent's pressure flag was on

In [ ]:
g = runs_ep.groupby(["team", "track_id"])
ps = g.agg(touches=("n_frames", "size"), frames_on_ball=("n_frames", "sum"),
           press_frames=("press_frames", "sum"), carry_prog_m=("carry_prog_m", "sum")).reset_index()
ps["time_on_ball_s"] = ps["frames_on_ball"] / FPS
ps["pct_on_ball_pressed"] = ps["press_frames"] / ps["frames_on_ball"]

made = (passes.groupby(["passer_team", "passer_id"])
        .agg(passes_made=("completed", "size"), passes_completed=("completed", "sum"),
             progressive_passes=("progressive_completed", "sum"), long_passes=("is_long", "sum"),
             passes_under_pressure=("under_pressure", "sum"), avg_pass_len_m=("length_m", "mean"))
        .reset_index().rename(columns={"passer_team": "team", "passer_id": "track_id"}))
recv = (passes[passes["completed"]].groupby(["receiver_team", "receiver_id"]).size()
        .rename("passes_received").reset_index().rename(columns={"receiver_team": "team", "receiver_id": "track_id"}))

player_buildup = ps.merge(made, on=["team", "track_id"], how="outer").merge(recv, on=["team", "track_id"], how="outer")
num_cols = [c for c in player_buildup.columns if c not in ("team", "track_id", "avg_pass_len_m")]
player_buildup[num_cols] = player_buildup[num_cols].fillna(0)
player_buildup["team"] = player_buildup["team"].astype(int)
player_buildup["touch_share"] = player_buildup["touches"] / player_buildup.groupby("team")["touches"].transform("sum")
player_buildup["pass_completion"] = player_buildup["passes_completed"] / player_buildup["passes_made"].replace(0, np.nan)
player_buildup["label"] = player_buildup["track_id"].map(LABEL)
player_buildup["is_gk"] = player_buildup["track_id"].map(IS_GK).fillna(False).astype(bool)
player_buildup["progressive_actions"] = player_buildup["progressive_passes"]   # carries reported separately in carry_prog_m
player_buildup = player_buildup.sort_values(["team", "touches"], ascending=[True, False]).reset_index(drop=True)

SHOW = ["label", "is_gk", "touches", "touch_share", "passes_made", "pass_completion", "passes_received",
        "progressive_passes", "long_passes", "carry_prog_m", "time_on_ball_s", "pct_on_ball_pressed"]
TOP_N = 10
for t in BUILD_TEAMS:
    print(f"\n=== {TEAM_NAMES[t]}: top {TOP_N} by build-up touches ===")
    display(player_buildup[player_buildup["team"] == t][SHOW].head(TOP_N)
            .round({"touch_share": 3, "pass_completion": 2, "carry_prog_m": 0, "time_on_ball_s": 1, "pct_on_ball_pressed": 2}).reset_index(drop=True))

In [ ]:
# who gets the ball and what they do with it (stacked: passes made + passes received)
fig, axes = plt.subplots(1, len(BUILD_TEAMS), figsize=(7 * len(BUILD_TEAMS), 5), sharex=True)
axes = np.atleast_1d(axes)
for ax, t in zip(axes, BUILD_TEAMS):
    d = player_buildup[player_buildup["team"] == t].head(TOP_N).iloc[::-1]
    ax.barh(d["label"], d["passes_made"], color=TEAM_COLORS[t], label="passes made")
    ax.barh(d["label"], d["passes_received"], left=d["passes_made"], color=TEAM_COLORS[t], alpha=0.4, label="passes received")
    for y, (_, r) in enumerate(d.iterrows()):
        ax.text(r["passes_made"] + r["passes_received"] + 0.5, y, f"{int(r['touches'])} touches" + (" (GK)" if r["is_gk"] else ""), va="center", fontsize=8)
    ax.set_title(f"{TEAM_NAMES[t]} - build-up involvement")
    ax.legend(loc="lower right", fontsize=8)
plt.tight_layout(); plt.show()

### Pass network (build-up passes only)
Node = player's average position while his team was building up (own attack frame, team attacks to the right, dashed line = 70% zone line), size = touches. Edge = completed passes between two players (>= 3), width = count.

In [ ]:
MIN_NODE_FRAMES = 6 * FPS
MIN_EDGE_PASSES = 3

_epf = ep_frames[["episode_id", "frame_idx"]].merge(episodes[["episode_id", "team"]], on="episode_id")
_pl = player_frame_table.loc[player_frame_table["team"].notna() & player_frame_table["pitch_x"].notna() & (player_frame_table["role"] != "referee"),
                             ["frame_idx", "track_id", "team", "pitch_x", "pitch_y"]]
_pl = _pl.merge(_epf.rename(columns={"team": "builder"}), on="frame_idx")
_pl = _pl[_pl["team"] == _pl["builder"]].copy()
_pl["x_att"], _pl["y_att"] = to_team_frame(_pl["team"].values, _pl["pitch_x"].values, _pl["pitch_y"].values)
avg_pos = _pl.groupby(["team", "track_id"]).agg(x=("x_att", "mean"), y=("y_att", "mean"), n=("x_att", "size")).reset_index()
avg_pos = avg_pos[avg_pos["n"] >= MIN_NODE_FRAMES]

fig, axes = plt.subplots(1, len(BUILD_TEAMS), figsize=(8 * len(BUILD_TEAMS), 6))
axes = np.atleast_1d(axes)
for ax, t in zip(axes, BUILD_TEAMS):
    draw_pitch(ax)
    nodes = avg_pos[avg_pos["team"] == t].set_index("track_id")
    touches = player_buildup[player_buildup["team"] == t].set_index("track_id")["touches"]
    e = (passes[(passes["passer_team"] == t) & passes["completed"]]
         .groupby(["passer_id", "receiver_id"]).size().reset_index(name="n"))
    e = e[(e["n"] >= MIN_EDGE_PASSES) & e["passer_id"].isin(nodes.index) & e["receiver_id"].isin(nodes.index)]
    for r in e.itertuples():
        a, b = nodes.loc[r.passer_id], nodes.loc[r.receiver_id]
        ax.plot([a["x"], b["x"]], [a["y"], b["y"]], color=TEAM_COLORS[t], alpha=0.45, lw=0.8 + 0.5 * r.n, zorder=2)
    for tid, r in nodes.iterrows():
        ax.scatter(r["x"], r["y"], s=60 + 25 * touches.get(tid, 0), color=TEAM_COLORS[t], ec="white", lw=1.5, zorder=3)
        num = NUMBER.get(tid, np.nan)
        ax.text(r["x"], r["y"], "" if pd.isna(num) else str(int(float(num))), ha="center", va="center", color="white", fontsize=8, zorder=4, weight="bold")
    ax.set_title(f"{TEAM_NAMES[t]} build-up pass network (attacks ->)")
plt.tight_layout(); plt.show()

# Part B — How each team builds up
**Per-frame episode table** (`epf`): one row per frame inside an episode, with the ball position in the builder's attack frame, who carries it, and whether the defending team is pressing (v30 flag, plus the proximity-only flag).

In [ ]:
epf = ep_frames[["episode_id", "frame_idx", "defending_team", "f_under", "f_prox_only", "f_both", "n_pressing"]].copy()
epf = epf.merge(episodes[["episode_id", "team"]].rename(columns={"team": "builder"}), on="episode_id")
epf = epf.rename(columns={"defending_team": "defender"})
assert epf["frame_idx"].is_unique, "a frame belongs to two episodes"
epf["f_active"] = epf["f_prox_only"] | epf["f_both"]

_b = ball_xy.reindex(epf["frame_idx"].values)
epf["ball_x"], epf["ball_y"] = _b["ball_pitch_x"].values, _b["ball_pitch_y"].values
epf["x_att"], epf["y_att"] = to_team_frame(epf["builder"].values, epf["ball_x"].values, epf["ball_y"].values)
epf["lane"], epf["band"] = lane_of(epf["y_att"]), band_of(epf["x_att"])

_c = ball_frame_table.drop_duplicates("frame_idx").set_index("frame_idx")["carrier_track_id"].reindex(epf["frame_idx"].values)
epf["carrier_id"] = _c.values
epf["carrier_type"] = np.where(pd.isna(epf["carrier_id"]), "ball in flight / loose",
                        np.where(pd.Series(epf["carrier_id"]).map(lambda v: IS_GK.get(int(v), False) if pd.notna(v) else False), "goalkeeper", "outfield player"))

# episode-level strategy features
_first = runs_ep.sort_values("start_frame").groupby("episode_id").first()
episodes["first_carrier_id"] = episodes["episode_id"].map(_first["track_id"])
episodes["first_carrier_is_gk"] = episodes["first_carrier_id"].map(lambda v: bool(IS_GK.get(int(v), False)) if pd.notna(v) else False)
episodes["start_type"] = np.where(episodes["first_carrier_is_gk"], "goalkeeper build-up", "open play")

_sb, _eb = ball_xy.reindex(episodes["start_frame"].values), ball_xy.reindex(episodes["end_frame"].values)
episodes["start_x_att"], episodes["start_y_att"] = to_team_frame(episodes["team"].values, _sb["ball_pitch_x"].values, _sb["ball_pitch_y"].values)
episodes["end_x_att"], episodes["end_y_att"] = to_team_frame(episodes["team"].values, _eb["ball_pitch_x"].values, _eb["ball_pitch_y"].values)
episodes["start_lane"], episodes["end_lane"] = lane_of(episodes["start_y_att"]), lane_of(episodes["end_y_att"])
episodes["start_band"], episodes["end_band"] = band_of(episodes["start_x_att"]), band_of(episodes["end_x_att"])

_pe = passes.groupby("episode_id").agg(n_passes=("completed", "size"), n_completed=("completed", "sum"), n_long=("is_long", "sum"),
                                       n_progressive=("progressive_completed", "sum"))
episodes = episodes.drop(columns=[c for c in _pe.columns if c in episodes.columns]).merge(_pe, left_on="episode_id", right_index=True, how="left")
for c in _pe.columns:
    episodes[c] = episodes[c].fillna(0).astype(int)
_last = passes.sort_values("release_frame").groupby("episode_id").last()
episodes["last_pass_long"] = episodes["episode_id"].map(_last["is_long"]).fillna(False).astype(bool)
episodes["last_pass_dir"] = episodes["episode_id"].map(_last["direction"])

print(f"{len(epf)} episode frames | carrier on the ball in {(epf['carrier_type'] != 'ball in flight / loose').mean():.0%} of them")
print("episode start types:", episodes.groupby(["team_name", "start_type"]).size().to_dict())

In [ ]:
# ---------- team strategy summary ----------
rows = []
for t in BUILD_TEAMS:
    e, p = episodes[episodes["team"] == t], passes[passes["passer_team"] == t]
    pc = p[p["completed"]]
    esc = e[e["outcome"] == "escaped"]
    rows.append({
        "team": TEAM_NAMES[t],
        "episodes": len(e),
        "escaped %": 100 * (e["outcome"] == "escaped").mean(),
        "lost %": 100 * (e["outcome"] == "lost").mean(),
        "goalkeeper-start %": 100 * e["first_carrier_is_gk"].mean(),
        "mean episode sec": e["duration_sec"].mean(),
        "passes / episode": len(p) / max(len(e), 1),
        "pass completion %": 100 * p["completed"].mean(),
        "mean pass length m": p["length_m"].mean(),
        "long balls % (>=30m)": 100 * p["is_long"].mean(),
        "forward %": 100 * (p["direction"] == "forward").mean(),
        "lateral %": 100 * (p["direction"] == "lateral").mean(),
        "backward %": 100 * (p["direction"] == "backward").mean(),
        "progressive passes / episode": p["progressive_completed"].sum() / max(len(e), 1),
        "passes from GK %": 100 * p["passer_is_gk"].mean(),
        "passes under pressure %": 100 * p["under_pressure"].mean(),
        "median passes before escape": esc["n_passes"].median(),
        "median sec to escape": esc["duration_sec"].median(),
        "escapes ending in a long ball %": 100 * esc["last_pass_long"].mean() if len(esc) else np.nan,
    })
strategy = pd.DataFrame(rows).set_index("team").T.round(1)
display(strategy)

print("episodes by start type and outcome (% within start type):")
display((pd.crosstab([episodes["team_name"], episodes["start_type"]], episodes["outcome"], normalize="index") * 100).round(0))

In [ ]:
# ---------- lanes and distance bands ----------
for t in BUILD_TEAMS:
    p = passes[(passes["passer_team"] == t) & passes["completed"]]
    print(f"\n=== {TEAM_NAMES[t]}: completed passes, release lane (rows) -> reception lane (columns), % of row ===")
    display((pd.crosstab(p["rel_lane"], p["rec_lane"], normalize="index").reindex(index=LANES, columns=LANES) * 100).round(0))

usage = []
for t in BUILD_TEAMS:
    r = runs_ep[runs_ep["team"] == t]
    b = pd.Series(band_of(r["start_x_att"])).value_counts(normalize=True).reindex(BANDS).fillna(0) * 100
    l = pd.Series(lane_of(r["start_y_att"])).value_counts(normalize=True).reindex(LANES).fillna(0) * 100
    usage.append(pd.concat([b, l], keys=["band", "lane"]).rename(TEAM_NAMES[t]))
print("\nwhere the touches happen (% of build-up touches):")
display(pd.concat(usage, axis=1).round(0))

In [ ]:
# ---------- how it ends: where each team escapes / loses the ball ----------
for t in BUILD_TEAMS:
    e = episodes[episodes["team"] == t]
    esc, lost = e[e["outcome"] == "escaped"], e[e["outcome"] == "lost"]
    print(f"\n=== {TEAM_NAMES[t]} ===")
    print(f"escaped: {len(esc)} | lost: {len(lost)} | other: {len(e) - len(esc) - len(lost)}")
    out = pd.concat({
        "escape lane %": esc["end_lane"].value_counts(normalize=True).reindex(LANES).fillna(0) * 100,
        "loss lane %": lost["end_lane"].value_counts(normalize=True).reindex(LANES).fillna(0) * 100,
    }, axis=1).round(0)
    display(out)
    display(pd.concat({"loss band %": lost["end_band"].value_counts(normalize=True).reindex(BANDS).fillna(0) * 100,
                       "start band % (all episodes)": e["start_band"].value_counts(normalize=True).reindex(BANDS).fillna(0) * 100}, axis=1).round(0))

fig, axes = plt.subplots(1, len(BUILD_TEAMS), figsize=(8 * len(BUILD_TEAMS), 6))
axes = np.atleast_1d(axes)
for ax, t in zip(axes, BUILD_TEAMS):
    draw_pitch(ax)
    d = epf[(epf["builder"] == t) & epf["x_att"].notna()]
    ax.hexbin(d["x_att"], d["y_att"], gridsize=24, extent=(0, PITCH_LENGTH, 0, PITCH_WIDTH), cmap="Greys", mincnt=1, alpha=0.65, zorder=1)
    e = episodes[(episodes["team"] == t) & episodes["end_x_att"].notna()]
    for outcome, col, mk in [("escaped", "#2ca02c", "o"), ("lost", "#d62728", "X")]:
        s = e[e["outcome"] == outcome]
        ax.scatter(s["end_x_att"], s["end_y_att"], c=col, marker=mk, s=45, ec="white", lw=0.6, zorder=3, label=f"{outcome} ({len(s)})")
    ax.legend(loc="lower right", fontsize=8)
    ax.set_title(f"{TEAM_NAMES[t]}: ball density in build-up + where episodes end (attacks ->)")
plt.tight_layout(); plt.show()

# Part C — How the opponent presses the build-up
For each builder team, the pressing team is the opponent. Pressure = the v30 `under_team_pressure` flag (proximity, closing speed, step-out, lane blocking; own-box frames excluded) from Part 0, unchanged.

**C1 — Where**: share of build-up frames under pressure by distance band x lane (builder's attack frame; cells with < 100 frames greyed out).

In [ ]:
MIN_CELL_FRAMES = 100
fig, axes = plt.subplots(2, len(BUILD_TEAMS), figsize=(6.5 * len(BUILD_TEAMS), 8))
axes = np.array(axes).reshape(2, len(BUILD_TEAMS))
for j, t in enumerate(BUILD_TEAMS):
    d = epf[(epf["builder"] == t) & epf["band"].notna() & epf["lane"].notna()]
    for i, (flag, ttl) in enumerate([("f_under", "any v30 pressure"), ("f_active", "active (proximity-driven) pressure")]):
        share = d.pivot_table(index="band", columns="lane", values=flag, aggfunc="mean").reindex(index=BANDS, columns=LANES)
        n = d.pivot_table(index="band", columns="lane", values=flag, aggfunc="size").reindex(index=BANDS, columns=LANES).fillna(0)
        arr = np.where(n.values >= MIN_CELL_FRAMES, share.values, np.nan)
        ax = axes[i, j]
        im = ax.imshow(arr * 100, cmap="Reds", vmin=0, vmax=60, aspect="auto")
        ax.set_xticks(range(3)); ax.set_xticklabels(LANES); ax.set_yticks(range(4)); ax.set_yticklabels(BANDS)
        for a in range(4):
            for b_ in range(3):
                if not np.isnan(arr[a, b_]):
                    ax.text(b_, a, f"{arr[a, b_] * 100:.0f}%\n(n={int(n.values[a, b_])})", ha="center", va="center", fontsize=8)
        ax.set_title(f"{TEAM_NAMES[opp_of[t]]} pressing {TEAM_NAMES[t]}'s build-up\n{ttl} (% of frames)", fontsize=9)
plt.tight_layout(); plt.show()

**C2 — On whom**: share of build-up frames under pressure, by who has the ball. Also how soon the first press arrives, split by how the episode started.

In [ ]:
print("pressure share by carrier type (% of frames):")
on_whom = (epf.groupby(["builder", "carrier_type"]).agg(frames=("f_under", "size"), pressed_pct=("f_under", "mean"), active_pct=("f_active", "mean")))
on_whom[["pressed_pct", "active_pct"]] *= 100
display(on_whom.rename(index=TEAM_NAMES, level=0).round(1))

print("\ntime to first press by start type (episodes that were pressed):")
pr = episodes[episodes["pressed"]]
display(pr.groupby(["team_name", "start_type"]).agg(episodes=("episode_id", "count"),
        median_sec_to_first_press=("time_to_first_press_sec", "median"), mean_pct_time_pressed=("pct_time_pressed", "mean")).round(2))
print("share of episodes pressed at all, by start type:")
display((episodes.groupby(["team_name", "start_type"])["pressed"].mean() * 100).round(0).to_frame("pressed %"))

# where the press STARTS (ball position at the first pressed frame)
fp = episodes[episodes["pressed"]].copy()
_p = ball_xy.reindex(fp["first_press_frame"].astype(int).values)
fp["px_att"], fp["py_att"] = to_team_frame(fp["team"].values, _p["ball_pitch_x"].values, _p["ball_pitch_y"].values)
fp["press_start_band"], fp["press_start_lane"] = band_of(fp["px_att"]), lane_of(fp["py_att"])
print("\nwhere the first press arrives (% of pressed episodes):")
display(pd.concat([fp.groupby("team_name")["press_start_band"].value_counts(normalize=True).mul(100).round(0).unstack().reindex(columns=BANDS),
                   fp.groupby("team_name")["press_start_lane"].value_counts(normalize=True).mul(100).round(0).unstack().reindex(columns=LANES)], axis=1))

**C3 — What triggers the press**: for every completed build-up pass, did the opponent's pressure follow within 2 s of reception? Rates with vs without each trigger (position/motion proxies, same idea as v30 section 16; there is no body-orientation model, and `gk_receives` is a pass to the goalkeeper).

In [ ]:
PRESS_ONSET_WINDOW_FRAMES = 2 * FPS
TOUCHLINE_DIST_THRESH_M = 8.0
ISOLATED_LANES_THRESH = 2
lane_open = frame_lane_summary.drop_duplicates("frame_idx").set_index("frame_idx")["n_lanes_open"]

rc = passes[passes["completed"]].copy()
rc["press_followed"] = [bool(under_arr[opp_of[int(t)]][int(f):int(f) + PRESS_ONSET_WINDOW_FRAMES + 1].any())
                        for t, f in zip(rc["passer_team"], rc["reception_frame"])]
rc["trigger_gk_receives"] = rc["receiver_is_gk"]
rc["trigger_backward_pass"] = rc["progression_m"] < 0
rc["trigger_touchline"] = np.minimum(rc["ry"], PITCH_WIDTH - rc["ry"]) < TOUCHLINE_DIST_THRESH_M
rc["trigger_deep_own_third"] = rc["rec_x_att"] < 35
rc["trigger_isolated"] = rc["reception_frame"].map(lane_open) < ISOLATED_LANES_THRESH
TRIGGERS = ["trigger_gk_receives", "trigger_backward_pass", "trigger_touchline", "trigger_deep_own_third", "trigger_isolated"]

out = []
for t in BUILD_TEAMS:
    r = rc[rc["passer_team"] == t]
    for col in TRIGGERS:
        m = r[col].fillna(False).astype(bool)
        out.append({"pressing team": TEAM_NAMES[opp_of[t]], "on build-up of": TEAM_NAMES[t], "trigger": col.replace("trigger_", ""),
                    "receptions with trigger": int(m.sum()), "press follows % (with)": 100 * r.loc[m, "press_followed"].mean() if m.any() else np.nan,
                    "press follows % (without)": 100 * r.loc[~m, "press_followed"].mean() if (~m).any() else np.nan})
    out.append({"pressing team": TEAM_NAMES[opp_of[t]], "on build-up of": TEAM_NAMES[t], "trigger": "(baseline: any reception)",
                "receptions with trigger": len(r), "press follows % (with)": 100 * r["press_followed"].mean(), "press follows % (without)": np.nan})
trig_tbl = pd.DataFrame(out)
trig_tbl["lift (pts)"] = trig_tbl["press follows % (with)"] - trig_tbl["press follows % (without)"]
display(trig_tbl.round(1).set_index(["pressing team", "on build-up of", "trigger"]))

**C4 — Who presses**: the defenders that actually close down the carrier (`is_pressuring` from v30 section 1) during the opponent's build-up episodes, how many press at once, and the height they press from.

In [ ]:
_ob = off_ball_pressure[["frame_idx", "team", "track_id", "is_pressuring", "pitch_x", "pitch_y"]]
_ob = _ob.merge(epf[["frame_idx", "episode_id", "defender"]], on="frame_idx")
_ob = _ob[_ob["team"] == _ob["defender"]]
_ob["dog"] = own_goal_distance(_ob["team"].values, _ob["pitch_x"].values)

pressers = (_ob.groupby(["defender", "track_id"])
            .agg(frames_on_pitch=("is_pressuring", "size"), pressing_frames=("is_pressuring", "sum"))
            .reset_index())
_hp = _ob[_ob["is_pressuring"]].groupby(["defender", "track_id"])["dog"].mean().rename("avg_height_when_pressing_m")
_ep_n = _ob[_ob["is_pressuring"]].groupby(["defender", "track_id"])["episode_id"].nunique().rename("episodes_pressed_in")
pressers = pressers.merge(_hp, on=["defender", "track_id"], how="left").merge(_ep_n, on=["defender", "track_id"], how="left")
pressers["share_of_team_pressing"] = pressers["pressing_frames"] / pressers.groupby("defender")["pressing_frames"].transform("sum")
pressers["label"] = pressers["track_id"].map(LABEL)
for d_ in BUILD_TEAMS:
    print(f"\n=== {TEAM_NAMES[d_]} pressers (while {TEAM_NAMES[opp_of[d_]]} builds up) ===")
    display(pressers[pressers["defender"] == d_].sort_values("pressing_frames", ascending=False)
            [["label", "pressing_frames", "share_of_team_pressing", "episodes_pressed_in", "avg_height_when_pressing_m"]].head(8).round(2).reset_index(drop=True))

print("\nnumber of pressers at once (share of PRESSED build-up frames):")
_np = epf[epf["f_under"]].copy()
_np["n_pressing_c"] = np.clip(_np["n_pressing"].fillna(0), 0, 4).astype(int).astype(str).replace({"4": "4+"})
display((_np.groupby(["defender", "n_pressing_c"]).size().groupby(level=0).transform(lambda s: 100 * s / s.sum()).unstack().round(0)).rename(index=TEAM_NAMES))

**C5 — Shape: block height, compactness, and man-oriented vs zonal**
- Block metrics use the defending team's outfield players per frame, split by *pressed* vs *not pressed* build-up frames: height = mean distance from own goal, depth = front-3 minus back-3, width = 10th-90th percentile of y.
- Man-oriented vs zonal (heuristic): in pressed frames, count builder players (excluding the carrier) that have a defender within 3 m (`n_marked`), and how often the same builder-defender pair is still the closest pair 1 s later (`pair_stability`). High both = man-oriented; low both = zonal / ball-oriented.

In [ ]:
players_ep = (player_frame_table.loc[player_frame_table["role"].isin(["player", "goalkeeper"]) & player_frame_table["team"].notna() & player_frame_table["pitch_x"].notna(),
                                     ["frame_idx", "track_id", "team", "role", "pitch_x", "pitch_y"]]
              .merge(epf[["frame_idx", "episode_id", "builder", "defender", "f_under", "carrier_id"]], on="frame_idx", how="inner"))

# ---------- block height / depth / width ----------
dd = players_ep[(players_ep["role"] == "player") & (players_ep["team"] == players_ep["defender"])].copy()
dd["dog"] = own_goal_distance(dd["team"].values, dd["pitch_x"].values)
dd = dd.sort_values(["frame_idx", "dog"])
dd["rk"] = dd.groupby("frame_idx").cumcount()
dd["n"] = dd.groupby("frame_idx")["dog"].transform("size")
dd = dd[dd["n"] >= 6]
blk = pd.concat([
    dd.groupby("frame_idx")["dog"].mean().rename("height_m"),
    dd[dd["rk"] < 3].groupby("frame_idx")["dog"].mean().rename("back3"),
    dd[dd["rk"] >= dd["n"] - 3].groupby("frame_idx")["dog"].mean().rename("front3"),
    (dd.groupby("frame_idx")["pitch_y"].quantile(0.9) - dd.groupby("frame_idx")["pitch_y"].quantile(0.1)).rename("width_m"),
], axis=1)
blk["depth_m"] = blk["front3"] - blk["back3"]
blk = blk.join(epf.set_index("frame_idx")[["defender", "f_under"]])
shape_tbl = (blk.groupby(["defender", "f_under"])[["height_m", "depth_m", "width_m"]].mean()
             .assign(frames=blk.groupby(["defender", "f_under"]).size()).round(1))
shape_tbl.index = pd.MultiIndex.from_tuples([(TEAM_NAMES[a], "pressing" if b else "not pressing") for a, b in shape_tbl.index], names=["defending team", "build-up frames"])
print("defending block shape:")
display(shape_tbl)

# ---------- man-oriented vs zonal ----------
MARK_DIST_M = 3.0
MARK_LAG_FRAMES = FPS
mk = players_ep[players_ep["f_under"] & (players_ep["role"] == "player")]
assign, nm_rows = {}, []
for f, g in mk.groupby("frame_idx", sort=True):
    b_team = g["builder"].iat[0]
    a = g[(g["team"] == b_team) & (g["track_id"] != g["carrier_id"])]
    d = g[g["team"] != b_team]
    if len(a) < 3 or len(d) < 3:
        continue
    D = np.hypot(a["pitch_x"].values[:, None] - d["pitch_x"].values[None, :], a["pitch_y"].values[:, None] - d["pitch_y"].values[None, :])
    j = D.argmin(axis=1)
    ok = D[np.arange(len(a)), j] < MARK_DIST_M
    assign[f] = dict(zip(a["track_id"].values[ok], d["track_id"].values[j[ok]]))
    nm_rows.append((f, int(g["defender"].iat[0]), int(ok.sum())))
nm = pd.DataFrame(nm_rows, columns=["frame_idx", "defender", "n_marked"])
_def_by_frame = nm.set_index("frame_idx")["defender"]
st_rows = []
for f, pairs in assign.items():
    prev = assign.get(f - MARK_LAG_FRAMES)
    if prev:
        common = [k for k in pairs if k in prev]
        if common:
            st_rows.append((int(_def_by_frame[f]), float(np.mean([pairs[k] == prev[k] for k in common])), len(common)))
stab = pd.DataFrame(st_rows, columns=["defender", "stable", "n"])
_stab = pd.Series({d_: float(np.average(g["stable"], weights=g["n"])) for d_, g in stab.groupby("defender")}, name="pair_stability", dtype=float)
marking = pd.concat([
    nm.groupby("defender")["n_marked"].agg(mean_n_marked="mean", frames="size"),
    nm.groupby("defender")["n_marked"].apply(lambda s: float((s >= 3).mean())).rename("share_frames_>=3_marked"),
    _stab,
], axis=1).rename(index=TEAM_NAMES).round(2)
print("\nman-oriented vs zonal (pressed build-up frames):")
display(marking)

**C6 — Pressing shape picture**: average positions (builder's attack frame, builder attacks ->) of both teams over frames where the build-up was **pressed** vs **not pressed**. Circles = building team, squares = pressing team, numbers = shirt numbers.

In [ ]:
MIN_SHAPE_FRAMES = 100
fig, axes = plt.subplots(len(BUILD_TEAMS), 2, figsize=(16, 6 * len(BUILD_TEAMS)))
axes = np.array(axes).reshape(len(BUILD_TEAMS), 2)
for i, t in enumerate(BUILD_TEAMS):
    sub = players_ep[players_ep["builder"] == t].copy()
    sub["x_att"], sub["y_att"] = to_team_frame(np.full(len(sub), t), sub["pitch_x"].values, sub["pitch_y"].values)
    avg = (sub.groupby(["f_under", "track_id", "team"]).agg(x=("x_att", "mean"), y=("y_att", "mean"), n=("x_att", "size")).reset_index())
    avg = avg[avg["n"] >= MIN_SHAPE_FRAMES]
    for j, pressed in enumerate([True, False]):
        ax = axes[i, j]; draw_pitch(ax)
        for r in avg[avg["f_under"] == pressed].itertuples():
            is_builder = r.team == t
            ax.scatter(r.x, r.y, s=260, marker="o" if is_builder else "s", color=TEAM_COLORS[int(r.team)], ec="white", lw=1.5, zorder=3, alpha=0.95)
            num = NUMBER.get(r.track_id, np.nan)
            ax.text(r.x, r.y, "" if pd.isna(num) else str(int(float(num))), ha="center", va="center", color="white", fontsize=8, weight="bold", zorder=4)
        ax.set_title(f"{TEAM_NAMES[t]} building (o) vs {TEAM_NAMES[opp_of[t]]} pressing (s) - {'PRESSED' if pressed else 'NOT pressed'} frames")
plt.tight_layout(); plt.show()

# Part D — Side-by-side summary per team
How each team builds, and how the opponent presses it. Numbers are computed live from the tables above; read them as descriptions of this half, not as causal effects (pressed episodes are longer than unpressed ones, so pressed-vs-unpressed outcome comparisons are biased toward long episodes — use the first-3-seconds flag for a fair comparison).

In [ ]:
def _pct(x): return f"{100 * x:.0f}%"
lines = []
for t in BUILD_TEAMS:
    e, p = episodes[episodes["team"] == t], passes[passes["passer_team"] == t]
    pb = player_buildup[player_buildup["team"] == t]
    top = pb.head(3)
    esc = e[e["outcome"] == "escaped"]
    d_ = opp_of[t]
    md_ = marking.loc[TEAM_NAMES[d_]] if TEAM_NAMES[d_] in marking.index else None
    on_gk = epf[(epf["builder"] == t) & (epf["carrier_type"] == "goalkeeper")]
    on_of = epf[(epf["builder"] == t) & (epf["carrier_type"] == "outfield player")]
    top_pr = pressers[pressers["defender"] == d_].sort_values("pressing_frames", ascending=False).head(3)
    first_pressed = e[e["pressed"]]["time_to_first_press_sec"].median()
    hot = (epf[(epf["builder"] == t) & epf["band"].notna()].groupby(["band", "lane"])["f_under"].agg(["mean", "size"]).query("size >= @MIN_CELL_FRAMES")["mean"])
    hot_txt = f"{hot.idxmax()[0]} / {hot.idxmax()[1]} ({_pct(hot.max())} of frames)" if len(hot) else "n/a"
    lines.append(f"""### {TEAM_NAMES[t]} build-up (pressed by {TEAM_NAMES[d_]})
- **Builds**: {len(e)} episodes; {_pct((e.outcome == 'escaped').mean())} escape the 70% line, {_pct((e.outcome == 'lost').mean())} end in a lost ball. {_pct(e.first_carrier_is_gk.mean())} start with the goalkeeper. {len(p) / max(len(e), 1):.1f} passes per episode, mean length {p.length_m.mean():.0f} m, {_pct(p.is_long.mean())} long balls; direction forward/lateral/backward = {_pct((p.direction == 'forward').mean())}/{_pct((p.direction == 'lateral').mean())}/{_pct((p.direction == 'backward').mean())}.
- **Escapes**: {('%d of %d episodes; median %.0f passes and %.0f s; %s end with a long ball; most go through the %s lane.' % (len(esc), len(e), esc.n_passes.median(), esc.duration_sec.median(), _pct(esc.last_pass_long.mean()), esc.end_lane.value_counts().idxmax())) if len(esc) else 'none in this sample.'}
- **Most involved**: {', '.join(f"{r.label} ({int(r.touches)} touches, {_pct(r.touch_share)})" for r in top.itertuples())}.
- **Pressed by {TEAM_NAMES[d_]}**: {_pct(e.pressed.mean())} of episodes pressed, first press after a median {first_pressed:.1f} s; {_pct(on_gk.f_under.mean()) if len(on_gk) else 'n/a'} of goalkeeper-on-ball frames vs {_pct(on_of.f_under.mean()) if len(on_of) else 'n/a'} of outfield-on-ball frames are pressed; most pressed zone: {hot_txt}.
- **Pressers**: {', '.join(f"{r.label} ({_pct(r.share_of_team_pressing)})" for r in top_pr.itertuples())}. Marking: {'%.1f players marked on average, pair stability %s' % (md_['mean_n_marked'], md_['pair_stability']) if md_ is not None else 'n/a'}.""")
display_md = "\n\n".join(lines)
try:
    from IPython.display import Markdown
    display(Markdown(display_md))
except Exception:
    print(display_md)

## Save

In [ ]:
OUT = CACHE / "buildup_vs_press"
OUT.mkdir(exist_ok=True)
episodes.to_parquet(OUT / "episodes.parquet", index=False)
passes.drop(columns=["rel_lane", "rec_lane", "rel_band", "rec_band"]).assign(rel_lane=passes["rel_lane"].astype(str), rec_lane=passes["rec_lane"].astype(str),
        rel_band=passes["rel_band"].astype(str), rec_band=passes["rec_band"].astype(str)).to_parquet(OUT / "buildup_passes.parquet", index=False)
player_buildup.to_parquet(OUT / "player_buildup.parquet", index=False)
pressers.to_parquet(OUT / "pressers.parquet", index=False)
print("saved to", OUT)